# ⚙️ SÍNTESE 2 — FASE DE OTIMIZAÇÃO
## *Notebooks 11 a 15 | Frentes 1 a 5: Tornando o modelo mais forte*

---

> **💡 Para qualquer leitor:** Na Fase 1, descobrimos o que funciona. Aqui, **afinamos cada parafuso** para que o sistema resista ao mundo real: ruído intenso, sensores diferentes e poucas anomalias para treinar.

---

## 📋 CRITÉRIOS DE ACEITAÇÃO (os mesmos desde o início!)

| Critério | Limite mínimo | Por quê importa no Edge? |
|----------|--------------|-------------------------|
| **pAUC@0.1 (Score DCASE)** | > 0.80 | Alarmes falsos custam dinheiro e confiança |
| **Latência de Inferência** | < 50 ms | Alerta em tempo real exige resposta rápida |
| **Memória Usada** | < 4 MB | Microcontroladores têm RAM limitada |

---

## 🗂️ O QUE ESTA SÍNTESE COBRE

```
Notebook 11 → Frente 1: Regularização L2 + Limiar Gamma
Notebook 12 → Frente 2: CNN + GMM (modelagem probabilística)
Notebook 13 → Frente 3: HHT + UKF (limpeza e isolamento de sinal)
Notebook 14 → Frente 4: Data Augmentation via Mixup
Notebook 15 → Frente 5: pAUC@0.1 + Score DCASE como métrica principal
```

**Bases de dados nesta fase:**
- 🎵 DCASE 2025 (Task 2 — First-Shot Unsupervised)
- 🏭 MIMII (múltiplos SNRs: -6dB, 0dB, +6dB)
- ⚙️ Kaggle Bearing
- 📱 Drive Próprio
- Protocolo: **LOSO** com anti-leakage rigoroso


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import gamma
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor'] = '#161b22'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = 'white'
plt.rcParams['xtick.color'] = 'white'
plt.rcParams['ytick.color'] = 'white'
plt.rcParams['axes.edgecolor'] = '#30363d'
plt.rcParams['grid.color'] = '#30363d'

print('✅ Fase 2: Otimização. Afinando o sistema!')

In [ ]:
# ============================================================
# VISUALIZAÇÃO — Linha do tempo das 5 Frentes
# ============================================================
fig, ax = plt.subplots(figsize=(16, 4))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#161b22')

frentes = [
    (11, 'NB 11\nFrente 1\nRegularização\nGamma', '#1f6feb', '✅'),
    (12, 'NB 12\nFrente 2\nCNN + GMM', '#9e6a03', '⚠️'),
    (13, 'NB 13\nFrente 3\nHHT + UKF', '#2ea043', '✅'),
    (14, 'NB 14\nFrente 4\nMixup', '#2ea043', '✅'),
    (15, 'NB 15\nFrente 5\npAUC + DCASE', '#2ea043', '✅'),
]

for i, (x, label, color, symbol) in enumerate(frentes):
    ax.barh(0, 2.8, left=(i * 3.2), color=color, alpha=0.85, height=0.6)
    ax.text(i * 3.2 + 1.4, 0, label, ha='center', va='center', fontsize=8,
            color='white', fontweight='bold')

ax.set_xlim(-0.3, 16.5)
ax.set_ylim(-0.8, 1.2)
ax.set_title('📅 Fase 2 — Frentes de Otimização (Notebooks 11 a 15)', 
             fontsize=13, color='white', pad=10)
ax.axis('off')

legend_elements = [
    mpatches.Patch(color='#2ea043', label='✅ Incorporado à arquitetura'),
    mpatches.Patch(color='#1f6feb', label='✅ Parcialmente incorporado'),
    mpatches.Patch(color='#9e6a03', label='⚠️ CNN descartada / GMM mantido'),
]
ax.legend(handles=legend_elements, loc='lower center', bbox_to_anchor=(0.5, -0.25),
          ncol=3, frameon=False, fontsize=9, labelcolor='white')

plt.tight_layout()
plt.show()

---
# 📖 NOTEBOOK 11 — Frente 1: Regularização L2 + Limiar Gamma
### *Como evitar que o modelo "decore" em vez de aprender*

## 🧒 Explicação simples

Imagine um aluno que memoriza todas as respostas do livro, mas quando vem uma pergunta nova, não sabe responder. Isso se chama **overfitting** (sobreajuste).

A **Regularização L2** é como um professor que proíbe o aluno de memorizar — ele precisa entender a regra geral.

O **Limiar Gamma** é a linha que separa "normal" de "anomalia". Em vez de escolher essa linha no olho ("parece que 0.5 é um bom número"), usamos uma fórmula matemática (distribuição Gamma) que se ajusta automaticamente aos dados.

## 🔬 Como funciona o Limiar Gamma?

```
1. Calcule o erro de reconstrução de TODOS os sons normais
2. Ajuste uma curva Gamma a esses erros
3. O limiar = percentil 95% dessa curva
4. Qualquer erro acima do limiar → ANOMALIA
```

## 📊 Impacto da Regularização L2 no XGBoost

| Configuração | pAUC@0.1 | Variância entre folds | Estabilidade |
|-------------|----------|----------------------|-------------|
| Sem L2 (padrão) | 0.89 | ±0.08 | Instável |
| **Com L2 (λ=1.0)** | **0.91** | **±0.03** | **Estável** |
| Com L2 (λ=10.0) | 0.87 | ±0.02 | Sub-otimizado |

**Melhor configuração:** `lambda=1.0` no XGBoost

## 🟢 DECISÃO: MANTIDO

> - Regularização L2 com λ=1.0 → incorporada ao XGBoost definitivamente
> - Limiar Gamma → estratégia formal para o regime não-supervisionado (expandida na Frente 6)
> - **Nenhuma penalidade em latência ou memória** (parâmetro, não componente novo)


In [ ]:
# ============================================================
# VISUALIZAÇÃO — Distribuição Gamma e Limiar Formal
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

# Distribuição Gamma
ax1 = axes[0]
ax1.set_facecolor('#161b22')
np.random.seed(42)
normal_errors = np.random.gamma(2, 0.3, 500)
anomaly_errors = np.random.gamma(5, 0.8, 50)
x_range = np.linspace(0, 8, 1000)
gamma_fit = gamma.pdf(x_range, a=2, scale=0.3)

ax1.hist(normal_errors, bins=40, color='#2ea043', alpha=0.7, density=True, label='Sons Normais')
ax1.hist(anomaly_errors, bins=20, color='#da3633', alpha=0.7, density=True, label='Sons Anômalos')
ax1.plot(x_range, gamma_fit, color='#1f6feb', linewidth=2.5, label='Curva Gamma ajustada')
threshold = gamma.ppf(0.95, a=2, scale=0.3)
ax1.axvline(threshold, color='#f0883e', linewidth=2.5, linestyle='--', label=f'Limiar Gamma (p95) = {threshold:.2f}')
ax1.set_title('🎯 Limiar Gamma: Separando Normal de Anômalo', color='white', fontsize=11)
ax1.set_xlabel('Erro de Reconstrução', color='white')
ax1.set_ylabel('Densidade', color='white')
ax1.legend(frameon=False, labelcolor='white', fontsize=9)

# Estabilidade com L2
ax2 = axes[1]
ax2.set_facecolor('#161b22')
lambdas = [0, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
pauc_vals = [0.89, 0.90, 0.905, 0.91, 0.905, 0.895, 0.87]
variance = [0.08, 0.06, 0.045, 0.03, 0.035, 0.025, 0.02]

ax2.plot(lambdas, pauc_vals, color='#2ea043', marker='o', linewidth=2.5, markersize=8, label='pAUC@0.1')
ax2.axvline(1.0, color='#f0883e', linestyle='--', linewidth=2, label='λ=1.0 (escolhido)')
ax2.set_title('🔩 Efeito da Regularização L2 no pAUC', color='white', fontsize=11)
ax2.set_xlabel('λ (força da regularização)', color='white')
ax2.set_ylabel('pAUC@0.1', color='white')
ax2.set_ylim(0.85, 0.95)
ax2.legend(frameon=False, labelcolor='white', fontsize=9)
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
# 📖 NOTEBOOK 12 — Frente 2: CNN + GMM
### *A rede neural que soube demais... e o modelo estatístico que salvou o dia*

## 🧒 Explicação simples

Imagine um grupo de moradores de um bairro. A "normalidade" desse bairro pode ter **vários perfis diferentes**: o padeiro que trabalha de manhã, o estudante que estuda à noite, a família que faz churrasco no domingo.

- **GMM** (Gaussian Mixture Model): modela esses **múltiplos perfis de normalidade** como uma mistura de gaussianas. Se um som não pertence a nenhum perfil → anomalia!
- **CNN** como extratora de features: processa o espectrograma para criar uma representação compacta

## 🔬 Por que o GMM é melhor que o modelo unimodal?

```
Modelo Unimodal (Mahalanobis): assume que o som normal
é sempre parecido (uma única "bola" no espaço de features).

GMM: assume que existem VÁRIOS tipos de som normal
(múltiplas "bolas"). Mais realista para máquinas que
operam em diferentes modos (alta/baixa carga, aquecimento, etc.)
```

## 📊 CNN × GMM no regime não-supervisionado

| Configuração | pAUC@0.1 | Latência | Memória | Observação |
|-------------|----------|----------|---------|------------|
| Mahalanobis (unimodal) | 0.82 | 5 ms | 0.2 MB | Baseline não-sup |
| **GMM (k=5 componentes)** | **0.88** | **20 ms** | **0.8 MB** | **Campeão não-sup** |
| CNN como classificador | 0.94 | 85 ms | 12 MB | Viola limites! |
| CNN embeddings + GMM | 0.91 | 92 ms | 12.8 MB | Viola limites! |

## 🔴 POR QUE A CNN COMO CLASSIFICADOR FOI DESCARTADA?

> **Critério violado: Latência (85ms) e Memória (12 MB)**
>
> A CNN mostrou bons números de pAUC, mas violou ambos os limites de Edge Computing.
> Além disso, com dados escassos típicos de AAD (poucas anomalias), a CNN tende a memorizar ruídos específicos do microfone em vez da assinatura da falha mecânica.
>
> **Decisão:**
> - CNN → Removida como classificador principal
> - CNN → Mantida como geradora de embeddings para XAI (Notebook 20)
> - **GMM com k=5** → Adotado como campeão do regime **não-supervisionado**

## 🟢 O QUE FOI MANTIDO

> O GMM com 5 componentes captura a multimodalidade da normalidade industrial sem custo proibitivo. Ele passa nos 3 critérios: pAUC 0.88, 20ms, 0.8MB.


In [ ]:
# ============================================================
# VISUALIZAÇÃO — GMM vs Mahalanobis: Multimodalidade
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

np.random.seed(42)
# Simular dois modos de operação normal
mode1 = np.random.multivariate_normal([1, 2], [[0.3, 0.1], [0.1, 0.3]], 150)
mode2 = np.random.multivariate_normal([4, 1], [[0.4, -0.1], [-0.1, 0.4]], 100)
anomalies = np.random.multivariate_normal([2.5, 4], [[0.2, 0], [0, 0.2]], 20)

for ax, title in zip(axes, 
    ['Mahalanobis (Unimodal): confunde normal com anômalo',
     'GMM (Multimodal ✅): distingue os modos de normalidade']):
    ax.set_facecolor('#161b22')
    ax.scatter(mode1[:, 0], mode1[:, 1], c='#2ea043', alpha=0.6, s=30, label='Normal Modo 1')
    ax.scatter(mode2[:, 0], mode2[:, 1], c='#1f6feb', alpha=0.6, s=30, label='Normal Modo 2')
    ax.scatter(anomalies[:, 0], anomalies[:, 1], c='#da3633', s=80, marker='X', label='Anomalias', zorder=5)
    
    if 'Mahalanobis' in title:
        # Uma grande elipse de decisão (unimodal)
        theta = np.linspace(0, 2*np.pi, 100)
        cx, cy = np.mean(np.vstack([mode1, mode2]), axis=0)
        ax.plot(cx + 2.5 * np.cos(theta), cy + 2.0 * np.sin(theta), 
                '--', color='#f0883e', linewidth=2, label='Fronteira Mahalanobis')
    else:
        # Duas elipses GMM (multimodal)
        theta = np.linspace(0, 2*np.pi, 100)
        ax.plot(1 + 1.0 * np.cos(theta), 2 + 0.9 * np.sin(theta),
                '--', color='#2ea043', linewidth=2, label='Fronteira GMM Modo 1')
        ax.plot(4 + 1.1 * np.cos(theta), 1 + 1.0 * np.sin(theta),
                '--', color='#1f6feb', linewidth=2, label='Fronteira GMM Modo 2')

    ax.set_title(title, color='white', fontsize=9, fontweight='bold')
    ax.legend(frameon=False, labelcolor='white', fontsize=8, loc='upper right')
    ax.set_xlabel('Feature 1', color='white')
    ax.set_ylabel('Feature 2', color='white')

plt.suptitle('🔍 Por que GMM supera Mahalanobis em máquinas com múltiplos modos de operação?',
             color='white', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
# 📖 NOTEBOOK 13 — Frente 3: HHT + UKF
### *O filtro mágico que separa o barulho do ambiente da falha da máquina*

## 🧒 Explicação simples

Imagine que você está tentando ouvir alguém falar num estádio lotado durante um jogo. O **HHT+UKF** é como usar um fone de ouvido com cancelamento de ruído:

- **HHT** (Transformada de Hilbert-Huang): decompõe o som em camadas — separa os "soluços" da máquina (transientes impulsivos) do ruído constante do ambiente
- **UKF** (Filtro Unscented de Kalman): rastreia e estima a trajetória do sinal verdadeiro, filtrando o ruído estocástico

## 🔬 Como o HHT funciona?

```
Sinal bruto → EMD (Empirical Mode Decomposition)
    ↓
IMFs (Modos Intrínsecos de Frequência): c1, c2, ..., cn
    ↓ UKF filtra cada IMF
c1 = vibração impulsiva da falha (MANTIDO)
c2 = harmônico estrutural (MANTIDO)
c3..cn = ruído ambiental (REMOVIDO)
    ↓
Sinal limpo = c1 + c2
```

## 📊 Impacto do HHT+UKF nas métricas

| Configuração | pAUC@0.1 | SNR melhorado | Melhoria pAUC |
|-------------|----------|--------------|---------------|
| Sem filtro | 0.91 | - | - |
| Só HHT | 0.92 | +3.2 dB | +1.1% |
| Só UKF | 0.91 | +1.8 dB | +0.0% |
| **HHT + UKF combinados** | **0.94** | **+5.7 dB** | **+3.3%** |

## 🟢 DECISÃO: NÚCLEO DA PIPELINE

> O HHT+UKF tornou-se o **primeiro estágio** de toda a pipeline — obrigatório antes de qualquer extração de features.
>
> **Custo:** +3ms de latência (total: 15ms) — dentro dos limites.
> **Benefício:** +3.3% em pAUC, +5.7 dB de SNR, melhor generalização cross-sensor.

## 🔴 O QUE FOI DESCARTADO COM ESSE NOTEBOOK

> **Descartado:** Análise de sinais brutos sem pré-processamento de separação de componentes.
>
> **Razão:** Sem a limpeza do HHT+UKF, o modelo aprende características do microfone em vez de características da falha mecânica. Isso causa o domain shift catastrófico visto no Notebook 10.


In [ ]:
# ============================================================
# VISUALIZAÇÃO — HHT: Sinal Bruto vs Limpo
# ============================================================
fig, axes = plt.subplots(3, 1, figsize=(16, 9))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('🔬 HHT + UKF: Isolando a Falha do Ruído Ambiental', 
             color='white', fontsize=13, fontweight='bold')

np.random.seed(42)
t = np.linspace(0, 1, 16000)

# Sinal da falha (transiente impulsivo)
fault = np.zeros(len(t))
fault[5000:5100] = 3.0 * np.sin(2 * np.pi * 2000 * t[5000:5100]) * np.exp(-50 * (t[5000:5100] - t[5050]))
fault[9000:9100] = 2.5 * np.sin(2 * np.pi * 2000 * t[9000:9100]) * np.exp(-50 * (t[9000:9100] - t[9050]))

# Ruído ambiental (contínuo, estocástico)
ambient_noise = 0.8 * np.sin(2 * np.pi * 120 * t) + 0.5 * np.random.randn(len(t))

# Sinal bruto
raw = fault + ambient_noise

# Sinal "limpo" simulado após HHT+UKF
cleaned = fault + 0.05 * np.random.randn(len(t))

plots = [
    (raw, '⚠️ Sinal Bruto (com ruído ambiental)', '#8b949e'),
    (ambient_noise, '🔴 Ruído Ambiental Isolado (removido pelo HHT+UKF)', '#da3633'),
    (cleaned, '✅ Sinal Limpo: apenas a assinatura da falha', '#2ea043'),
]

for ax, (signal, title, color) in zip(axes, plots):
    ax.set_facecolor('#161b22')
    ax.plot(t, signal, color=color, linewidth=0.8, alpha=0.9)
    ax.set_title(title, color='white', fontsize=10, fontweight='bold')
    ax.set_xlim(0, 1)
    ax.set_ylabel('Amplitude', color='white', fontsize=9)
    ax.grid(alpha=0.2)

axes[-1].set_xlabel('Tempo (s)', color='white')
plt.tight_layout()
plt.show()
print('Os "picos" no sinal verde são os transientes da falha — invisíveis no sinal bruto!')

---
# 📖 NOTEBOOK 14 — Frente 4: Data Augmentation via Mixup
### *Como ensinar o modelo a não entrar em pânico quando vê dados novos*

## 🧒 Explicação simples

Imagine que você quer aprender a reconhecer cachorros. Mas você só viu cachorros amarelos. Quando vê um cachorro preto, você fica confuso.

O **Mixup** faz o seguinte: combina dois exemplos diferentes com uma "mistura" matemática:

```
Novo_som = α × Som_A + (1-α) × Som_B
Nova_label = α × Label_A + (1-α) × Label_B

Exemplo: α=0.7
Novo_som = 70% som normal + 30% som anômalo
Nova_label = [0.7 normal, 0.3 anômalo]  ("fronteira suavizada")
```

Isso força o modelo a aprender **regiões de transição** entre normal e anomalia, tornando-o mais robusto a variações.

## 📊 Impacto do Mixup no pAUC (especialmente em domain shift)

| Configuração | pAUC@0.1 (controlado) | pAUC@0.1 (cego/cross-domain) | Melhoria |
|-------------|----------------------|-----------------------------|---------|
| Sem augmentation | 0.94 | 0.51 | - |
| Com Mixup (α=0.2) | 0.93 | 0.62 | +21.6% |
| **Com Mixup (α=0.4)** | **0.94** | **0.68** | **+33.3%** |
| Com Mixup (α=0.8) | 0.91 | 0.65 | +27.5% |

## 🟢 DECISÃO: MANTIDO (α=0.4)

> O Mixup com α=0.4 tornou-se padrão no regime supervisionado. **Zero custo de latência ou memória** (é feito apenas durante o treino, não na inferência).
>
> Aumentou o pAUC em domain shift de 0.51 para 0.68 — diferença de 33% que torna o sistema mais confiável em campo.


In [ ]:
# ============================================================
# VISUALIZAÇÃO — Mixup: Interpolação entre classes
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0d1117')
fig.suptitle('🎨 Mixup: Criando Fronteiras Suaves entre Normal e Anômalo', 
             color='white', fontsize=12, fontweight='bold')

t = np.linspace(0, 1, 200)
normal = np.sin(2 * np.pi * 3 * t) * 0.5
anomaly = np.sin(2 * np.pi * 3 * t) * 0.5 + 2.0 * np.exp(-50*(t-0.5)**2)

configs = [
    (1.0, 0.0, '100% Normal', '#2ea043'),
    (0.6, 0.4, '60% Normal + 40% Anômalo (Mixup α=0.4)', '#9e6a03'),
    (0.0, 1.0, '100% Anômalo', '#da3633'),
]

for ax, (alpha, beta, title, color) in zip(axes, configs):
    ax.set_facecolor('#161b22')
    mixed = alpha * normal + beta * anomaly
    ax.plot(t, mixed, color=color, linewidth=2.5)
    ax.fill_between(t, mixed, alpha=0.3, color=color)
    ax.set_title(title, color=color, fontsize=10, fontweight='bold')
    ax.set_xlabel('Tempo (s)', color='white', fontsize=9)
    ax.set_ylabel('Amplitude', color='white', fontsize=9)
    ax.set_ylim(-1.2, 2.5)
    ax.grid(alpha=0.2)

plt.tight_layout()
plt.show()
print('O sinal amarelo do meio ensina o modelo a reconhecer estados "entre" normal e anômalo.')

---
# 📖 NOTEBOOK 15 — Frente 5: pAUC@0.1 + Score DCASE
### *Por que "90% de acurácia" NÃO é suficiente numa fábrica?*

## 🧒 Explicação simples

Imagine um detector de fumaça. Tem dois tipos de erro:
1. **Falso positivo**: alarme toca sem haver fumaça (todo mundo sai correndo à toa — prejuízo!)
2. **Falso negativo**: há fumaça mas o alarme não toca (a fábrica pega fogo — catástrofe!)

Na indústria, **falsos positivos são muito caros**: param a produção, desgastam a equipe e destroem a confiança no sistema.

Por isso, medimos **pAUC@0.1** = "quão bom é o detector quando a taxa de falso positivo é menor que 10%?"

## 🔬 O que é pAUC@FPR0.1?

```
Curva ROC: TPR (eixo Y) × FPR (eixo X)

AUC normal = área de toda a curva
pAUC@0.1  = área APENAS onde FPR < 0.1 (região industrial)
             (normalizado para ficar entre 0 e 1)
```

## 📊 Mudança de métrica: impacto no ranking dos modelos

| Modelo | F1-Score | AUC (global) | pAUC@0.1 | Ranking F1 | Ranking DCASE |
|--------|----------|-------------|----------|------------|---------------|
| GRU | 0.88 | 0.92 | 0.76 | 2º | 4º |
| Tiny-AST | 0.97 | 0.99 | 0.91 | 1º | 2º |
| **XGBoost + HHT** | **0.95** | **0.97** | **0.94** | **3º** | **1º** |
| GMM (não-sup) | 0.82 | 0.90 | 0.88 | 4º | 3º |

## ⚡ A VIRADA: O modelo campeão muda quando usamos pAUC!

> **Pela F1-Score global**, o Tiny-AST parecia campeão (0.97).
>
> **Pela pAUC@0.1 (DCASE)**, o XGBoost é campeão supervisionado (0.94) e o GMM é campeão não-supervisionado (0.88) — e ambos passam nos critérios de Edge!
>
> **Decisão:** A partir deste ponto, **pAUC@0.1 é a métrica principal** do projeto.

## 🔴 O QUE FOI DESCARTADO

> **Descartado:** AUC global como métrica de seleção de modelos
>
> **Descartado:** F1-Score como critério único de decisão
>
> **Razão:** Métricas globais mascaram comportamento ruim na região de baixo FPR, que é exatamente onde um sistema industrial opera.


In [ ]:
# ============================================================
# VISUALIZAÇÃO — Curva ROC e região pAUC@0.1
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d1117')

ax1 = axes[0]
ax1.set_facecolor('#161b22')

# Simular curvas ROC
fpr = np.linspace(0, 1, 100)
tpr_xgb = 1 - (1 - fpr)**3.5
tpr_gru = 1 - (1 - fpr)**2.8
tpr_tiny = 1 - (1 - fpr)**4.5

ax1.plot(fpr, tpr_xgb, color='#2ea043', linewidth=2.5, label='XGBoost (pAUC=0.94) ✅')
ax1.plot(fpr, tpr_gru, color='#1f6feb', linewidth=2.5, label='GRU (pAUC=0.76) ⚠️')
ax1.plot(fpr, tpr_tiny, color='#da3633', linewidth=2.5, linestyle='--', label='Tiny-AST (pAUC=0.91) 🔴 Edge')
ax1.fill_between(fpr[:11], tpr_xgb[:11], alpha=0.25, color='#2ea043', label='pAUC@0.1 (região medida)')
ax1.axvline(0.1, color='#f0883e', linewidth=2, linestyle='--', label='FPR = 0.1 (limite industrial)')
ax1.plot([0, 1], [0, 1], 'k--', alpha=0.4, linewidth=1)
ax1.set_title('🎯 Curva ROC: Por que pAUC@0.1 importa?', color='white', fontsize=11)
ax1.set_xlabel('FPR (Taxa de Falso Positivo)', color='white')
ax1.set_ylabel('TPR (Taxa de Verdadeiro Positivo)', color='white')
ax1.legend(frameon=False, labelcolor='white', fontsize=8)
ax1.set_xlim(0, 1); ax1.set_ylim(0, 1)

# Ranking por métrica
ax2 = axes[1]
ax2.set_facecolor('#161b22')
modelos = ['GRU', 'Tiny-AST', 'XGBoost\n+HHT', 'GMM']
f1_scores = [0.88, 0.97, 0.95, 0.82]
pauc_scores = [0.76, 0.91, 0.94, 0.88]
x = np.arange(len(modelos))
width = 0.35

bars1 = ax2.bar(x - width/2, f1_scores, width, label='F1-Score Global', color='#1f6feb', alpha=0.8)
bars2 = ax2.bar(x + width/2, pauc_scores, width, label='pAUC@0.1 (DCASE)', color='#2ea043', alpha=0.8)
ax2.axhline(0.80, color='#f0883e', linestyle='--', linewidth=2, label='Limite mínimo (0.80)')
ax2.set_xticks(x); ax2.set_xticklabels(modelos, color='white', fontsize=9)
ax2.set_ylim(0.7, 1.05)
ax2.set_title('⚡ A Virada: Ranking muda com pAUC vs F1', color='white', fontsize=11)
ax2.legend(frameon=False, labelcolor='white', fontsize=9)
ax2.set_ylabel('Score', color='white')

plt.tight_layout()
plt.show()

---
# 🏆 RESUMO FINAL — FASE DE OTIMIZAÇÃO
## *O que sobreviveu das 5 Frentes*

## ✅ INCORPORADOS À ARQUITETURA DEFINITIVA:

| Frente | Técnica | Impacto em pAUC | Custo Edge |
|--------|---------|----------------|------------|
| F1 | Regularização L2 (λ=1.0) no XGBoost | +2.2% | Zero |
| F1 | Limiar Gamma (distribuição formal) | Estabilidade | Zero |
| F2 | **GMM (k=5) como campeão não-sup** | 0.88 pAUC | 20ms / 0.8MB |
| F3 | **HHT + UKF como pré-processador** | +3.3% pAUC | +3ms / Zero mem |
| F4 | Mixup (α=0.4) no treinamento | +33% cross-domain | Zero (só treino) |
| F5 | **pAUC@0.1 como métrica principal** | Muda o ranking! | Zero |

## 🔴 DESCARTADOS NESTA FASE:

| Técnica | Motivo | Critério Violado |
|---------|--------|------------------|
| CNN como classificador | 85ms + 12MB | Latência + Memória |
| AUC global como métrica principal | Mascara FPR crítico | Score DCASE |
| F1-Score como critério único | Idem | Score DCASE |
| Análise de sinal bruto (sem HHT) | Domain shift catastrófico | Score DCASE |

---

## ➡️ PRÓXIMO PASSO: SINTESE_3_Validacao.ipynb
### *Frentes 6 a 11: Provando que o sistema funciona em condições reais*

---
*Síntese por: Emanoel Spanhol | Projeto AudioAlert | Pós-Graduação IA Aplicada — UniSENAI | 2025-2026*